# YOLO26 Orchid **Stage Detection** — Kaggle Training (v2)

Kaggle-ready version of the YOLO26 orchid stage-detection notebook. Trains on the same `YOLO26.zip` (or unzipped `YOLO26/` folder) you uploaded as a Kaggle Dataset.

**Why Kaggle (vs. Colab)**
- Free **P100 (16 GB)** or **T4 x2** — usually faster / more stable than Colab.
- **No mid-training disconnects** — use **Save & Run All (Commit)** and it runs in the background.
- Persistent `Output` tab — the trained `best.pt` and metrics survive after the run finishes.

**GPU note (important)**  
If you pick **Tesla P100** and see `CUDA error: no kernel image is available for execution on the device`, that means PyTorch was built **without Pascal (sm_60) kernels**. Cell 1 below auto-installs **`torch==2.5.1+cu121`** on P100/K80 so training works. Alternatively, switch the accelerator to **GPU T4 x2** — the default PyTorch wheel usually works there without extra steps.

**Before running this notebook (one-time setup)**
1. Phone-verify your Kaggle account → Settings → Phone Verification (required for GPU).
2. Top-nav → **Create → New Dataset** → upload `YOLO26.zip` (do **not** also upload loose `data.yaml` / `train` folders alongside the zip — that's what causes the "Conflicting files: data.yaml" error).
3. Create a new notebook → right sidebar:
   - **Accelerator** = GPU **P100** or **GPU T4 x2**
   - **Internet** = ON (needed to download `yolo26m.pt`)
   - **Persistence** = Variables and Files
   - Click **Add Data** → attach the dataset you uploaded.
4. **Run All** (or **Save & Run All**) — top-to-bottom.

**This notebook fixes the original notebook's problems**
1. The original had only **3 validation images, all of one class** → can't pick a "best" checkpoint that detects all stages.
2. Heavy **class imbalance** → rare stages were never learned (recall = 0).
3. **Stratified re-split** here guarantees every split contains every stage.
4. Stronger augmentation, AdamW + cosine LR, larger model, classification-weighted loss → pushes the model to actually discriminate stages.
5. Reports **per-class** P/R/F1/mAP + confusion matrix + a `predict_stage()` helper.

> Requires `ultralytics >= 8.4.0` for `yolo26*.pt` weights.

## 1. Install Ultralytics + verify GPU + Internet

In [ ]:
"""Set up YOLO26 training environment on Kaggle.

Strategy:

1. Audit what's loaded (informational only).
2. Always run pip install to make sure the **disk** has the right pinned versions.
   This is idempotent and harmless if disk is already correct.
3. Try to import. If a *wrong* numpy/PIL/torch is already loaded in this kernel
   from Kaggle's startup, the import will succeed for matching versions but fail
   for mismatched C extensions. In that case we explicitly tell the user to
   restart the kernel (now the disk has the right versions, so post-restart
   Kaggle's startup will load them).
"""
import importlib.metadata as _md
import socket
import subprocess
import sys

# --- 0. Internet check ----------------------------------------------------
for host in ("pypi.org", "github.com"):
    try:
        socket.gethostbyname(host)
    except Exception as exc:
        raise RuntimeError(
            f"Internet is OFF (DNS failed for {host}: {exc}).\n"
            "Right sidebar -> Internet -> ON. If the toggle is missing, "
            "verify your phone in Kaggle Settings."
        ) from exc

# --- 1. Detect GPU --------------------------------------------------------
try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=15, check=False,
    )
    gpu_line = (smi.stdout or "").strip().splitlines()[0] if smi.returncode == 0 else ""
except Exception:
    gpu_line = ""
print("GPU (nvidia-smi):", gpu_line or "(unknown)")

_needs_pascal_torch = any(
    s in gpu_line for s in ("P100", "K80", "M60", "P40", "Tesla P4")
)


# --- 2. Audit (info only) -------------------------------------------------
def _parse(v):
    parts = []
    for chunk in v.split("+", 1)[0].split("."):
        try:
            parts.append(int(chunk))
        except ValueError:
            break
    return tuple(parts)


def _audit():
    out = {}
    if "numpy" in sys.modules:
        v = getattr(sys.modules["numpy"], "__version__", "0")
        ok = _parse(v) >= (2, 1) and _parse(v) < (2, 2)
        out["numpy"] = ("ok" if ok else "wrong_loaded", v)
    else:
        out["numpy"] = ("not_loaded", None)
    if "PIL" in sys.modules:
        try:
            v = _md.version("pillow")
        except Exception:
            v = getattr(sys.modules["PIL"], "__version__", "0")
        ok = _parse(v) < (12, 0)
        out["pillow"] = ("ok" if ok else "wrong_loaded", v)
    else:
        out["pillow"] = ("not_loaded", None)
    if "torch" in sys.modules:
        torch_mod = sys.modules["torch"]
        v = getattr(torch_mod, "__version__", "0")
        try:
            cuda_ok = bool(torch_mod.cuda.is_available())
        except Exception:
            cuda_ok = False
        bad_pascal = _needs_pascal_torch and ("+cu128" in v or v.startswith("2.10"))
        ok = cuda_ok and not bad_pascal
        out["torch"] = ("ok" if ok else "wrong_loaded", v)
    else:
        out["torch"] = ("not_loaded", None)
    if "ultralytics" in sys.modules:
        try:
            out["ultralytics"] = ("ok", _md.version("ultralytics"))
        except Exception:
            out["ultralytics"] = ("ok", "?")
    else:
        out["ultralytics"] = ("not_loaded", None)
    return out


audit = _audit()
print("\nKernel audit:")
for pkg, (status, ver) in audit.items():
    print(f"  {pkg:<12} {status:<14} {ver or ''}")

wrong_loaded = [p for p, (s, _) in audit.items() if s == "wrong_loaded"]
all_ok = all(s == "ok" for s, _ in audit.values())


# --- 3. Make sure DISK has the right versions -----------------------------
PIN_NUMPY  = "numpy>=2.1,<2.2"   # 2.0 lacks `_center` symbol scipy needs; 2.2+ breaks numba/tf
PIN_PILLOW = "pillow>=10.0,<12.0"

if all_ok:
    print("\nAll required packages already loaded with good versions -> skipping pip.")
else:
    print("\nRunning pip install to fix DISK packages (kernel restart may follow)...")

    # Wipe stale numpy/pillow on disk so pip lays down clean files.
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "numpy", "pillow"],
        check=False,
    )

    cmd = [
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "ultralytics>=8.4.0",
        "scikit-learn>=1.3",
        "matplotlib>=3.7",
        "pandas>=2.0",
        "PyYAML>=6.0",
        PIN_NUMPY,
        PIN_PILLOW,
    ]
    if _needs_pascal_torch:
        print("Pascal GPU -> torch==2.5.1+cu121 (Pascal sm_60 kernels).")
        cmd += [
            "torch==2.5.1+cu121",
            "torchvision==0.20.1+cu121",
            "--extra-index-url", "https://download.pytorch.org/whl/cu121",
        ]
    else:
        cmd += ["torch", "torchvision"]
    subprocess.check_call(cmd)

    # Lock numpy/pillow versions so transitive deps cannot bump them.
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-deps", "--force-reinstall", "--no-cache-dir",
        PIN_NUMPY, PIN_PILLOW,
    ])

    # Drop noisy resolver warning from leftover torchaudio (Kaggle preinstalls it).
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"],
        check=False,
    )

# --- 4. If a wrong numpy/torch was already loaded, force a kernel restart -
if wrong_loaded:
    print()
    print("=" * 70)
    print("DISK now has the correct package versions, but this kernel still")
    print("has the OLD versions of " + ", ".join(wrong_loaded) + " loaded in memory.")
    print("Kaggle pre-imports numpy at kernel startup, so a *kernel restart*")
    print("(NOT just re-running the cell) is needed for the new versions to load.")
    print()
    print("ACTION:")
    print("  Top menu -> Run -> 'Restart session' (or click the circular-arrow")
    print("  refresh icon next to the cell run button).")
    print("  Then click 'Run All' from the top.")
    print("=" * 70)
    raise SystemExit(
        "Kernel restart required (disk is fixed, kernel still holds old C extensions)."
    )

# --- 5. Import + verify ---------------------------------------------------
import numpy as _np
from PIL import Image as _PILImage  # noqa: F401
import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Right sidebar -> Accelerator -> GPU P100 or GPU T4 x2."
    )

_x = torch.zeros(1, device="cuda")
del _x

print("\n--- versions ---")
print("ultralytics:", _md.version("ultralytics"))
print("torch       :", torch.__version__)
print("numpy       :", _np.__version__)
print("Pillow      :", _md.version("pillow"))
print("GPU         :", torch.cuda.get_device_name(0))
print("CUDA smoke  : OK")

DEVICE = 0

## 2. Locate the dataset

Looks under `/kaggle/input/` for any of:
1. A folder that already contains `data.yaml` + `train/` + (`valid/` or `val/`) — used directly.
2. A `YOLO26*.zip` (or any `*.zip`) — extracted into `/kaggle/working/yolo26_dataset_raw/`.

Works whether you uploaded the zip **or** the unpacked folder as your Kaggle Dataset.

In [ ]:
from pathlib import Path
import shutil
import zipfile

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT  = Path("/kaggle/working")

if not INPUT_ROOT.exists():
    raise RuntimeError(
        "/kaggle/input does not exist. Are you running this on Kaggle? "
        "If on Colab, use the Colab notebook instead."
    )

print("Available Kaggle datasets:")
for d in sorted(INPUT_ROOT.iterdir()):
    print("  -", d)

def _looks_like_yolo_root(folder: Path) -> bool:
    if not folder.is_dir():
        return False
    if not (folder / "data.yaml").exists():
        return False
    has_train = (folder / "train" / "images").is_dir()
    has_val = (folder / "valid" / "images").is_dir() or (folder / "val" / "images").is_dir()
    return has_train and has_val

EXTRACTED_DIR = None
ALREADY_UNPACKED = None

for ds in INPUT_ROOT.iterdir():
    if not ds.is_dir():
        continue
    if _looks_like_yolo_root(ds):
        ALREADY_UNPACKED = ds
        break
    for sub in ds.rglob("data.yaml"):
        if _looks_like_yolo_root(sub.parent):
            ALREADY_UNPACKED = sub.parent
            break
    if ALREADY_UNPACKED is not None:
        break

if ALREADY_UNPACKED is not None:
    print("\nFound an already-unpacked YOLO dataset at:", ALREADY_UNPACKED)
    EXTRACTED_DIR = ALREADY_UNPACKED
else:
    zip_candidates = list(INPUT_ROOT.rglob("YOLO26*.zip")) or list(INPUT_ROOT.rglob("*.zip"))
    if not zip_candidates:
        raise FileNotFoundError(
            "No YOLO dataset folder and no .zip found in /kaggle/input. "
            "Use 'Add Data' on the right sidebar to attach your YOLO26 dataset."
        )
    DATASET_ZIP = zip_candidates[0]
    print("\nUsing dataset zip:", DATASET_ZIP)

    EXTRACTED_DIR = WORK_ROOT / "yolo26_dataset_raw"
    if EXTRACTED_DIR.exists():
        shutil.rmtree(EXTRACTED_DIR)
    EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(EXTRACTED_DIR)
    print("Extracted to:", EXTRACTED_DIR)

print("\nEXTRACTED_DIR =", EXTRACTED_DIR)

## 3. Read the original `data.yaml` and gather every image + label

Combines `train/`, `valid/` (or `val/`) and `test/` into one pool so we can re-split correctly.

In [ ]:
import yaml
from collections import Counter
from pathlib import Path

candidate_yamls = list(EXTRACTED_DIR.rglob("data.yaml"))
if not candidate_yamls:
    raise FileNotFoundError("data.yaml not found inside the dataset.")

def _yaml_score(p: Path):
    d = p.parent
    has_train = (d / "train" / "images").is_dir()
    has_val = (d / "valid" / "images").is_dir() or (d / "val" / "images").is_dir()
    return (0 if has_train and has_val else 1, len(p.parts))

ORIG_YAML = sorted(candidate_yamls, key=_yaml_score)[0]
ORIG_DIR  = ORIG_YAML.parent
print("Original data.yaml:", ORIG_YAML)
print("Original dataset root:", ORIG_DIR)

with open(ORIG_YAML, "r") as f:
    orig_cfg = yaml.safe_load(f)

if isinstance(orig_cfg.get("names"), (list, tuple)):
    CLASS_NAMES = list(orig_cfg["names"])
else:
    CLASS_NAMES = [orig_cfg["names"][i] for i in sorted(orig_cfg["names"].keys())]
NUM_CLASSES = len(CLASS_NAMES)
print("Classes :", CLASS_NAMES)
print("nc      :", NUM_CLASSES)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
all_images = []
for split in ("train", "valid", "val", "test"):
    split_imgs = ORIG_DIR / split / "images"
    if not split_imgs.is_dir():
        continue
    for p in split_imgs.iterdir():
        if p.suffix.lower() in IMG_EXTS:
            all_images.append(p)

print("Total images found:", len(all_images))
if len(all_images) < 30:
    raise RuntimeError("Not enough images; expected at least 30.")

def _label_path_for(img: Path) -> Path:
    return img.parent.parent / "labels" / (img.stem + ".txt")

usable = []
class_counts_total = Counter()
images_without_labels = 0
empty_label_files = 0
for img in all_images:
    lbl = _label_path_for(img)
    if not lbl.exists():
        images_without_labels += 1
        continue
    classes_in_img = []
    for line in lbl.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            cid = int(line.split()[0])
        except Exception:
            continue
        if 0 <= cid < NUM_CLASSES:
            classes_in_img.append(cid)
    if not classes_in_img:
        empty_label_files += 1
        continue
    usable.append((img, lbl, classes_in_img))
    class_counts_total.update(classes_in_img)

print(f"Images with labels      : {len(usable)}")
print(f"Images without labels   : {images_without_labels}")
print(f"Empty label files       : {empty_label_files}")
print("\nObject counts per class (entire dataset):")
for cid in range(NUM_CLASSES):
    print(f"  {cid} {CLASS_NAMES[cid]:<16} : {class_counts_total[cid]}")

## 4. Stratified re-split into train / valid / test (80 / 10 / 10)

Each image is tagged with its **dominant class** (the most frequent class id in its label file). `train_test_split(..., stratify=...)` then guarantees every split contains every stage.

In [ ]:
from collections import Counter
import random
import shutil
import yaml
from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)

DATASET_DIR = Path("/kaggle/working/yolo26_dataset")
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
for split in ("train", "valid", "test"):
    (DATASET_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

dominant_labels = []
for img, lbl, classes_in_img in usable:
    dominant = Counter(classes_in_img).most_common(1)[0][0]
    dominant_labels.append(dominant)

paths_train, paths_temp, y_train, y_temp = train_test_split(
    usable, dominant_labels,
    test_size=0.20, random_state=SEED, stratify=dominant_labels,
)
paths_valid, paths_test, y_valid, y_test = train_test_split(
    paths_temp, y_temp,
    test_size=0.50, random_state=SEED, stratify=y_temp,
)

def _copy_to_split(items, split_name):
    counts = Counter()
    for img, lbl, classes_in_img in items:
        shutil.copy2(img, DATASET_DIR / split_name / "images" / img.name)
        shutil.copy2(lbl, DATASET_DIR / split_name / "labels" / lbl.name)
        counts.update(classes_in_img)
    return counts

print("Re-splitting (stratified by dominant class)...")
counts_train = _copy_to_split(paths_train, "train")
counts_valid = _copy_to_split(paths_valid, "valid")
counts_test  = _copy_to_split(paths_test,  "test")

print(f"\n{'split':<8}{'images':>8}  per-class object counts")
for split, items, counts in [
    ("train", paths_train, counts_train),
    ("valid", paths_valid, counts_valid),
    ("test",  paths_test,  counts_test),
]:
    per_class = "  ".join(
        f"{CLASS_NAMES[c]}={counts[c]}" for c in range(NUM_CLASSES)
    )
    print(f"{split:<8}{len(items):>8}  {per_class}")

missing = []
for split, counts in [("train", counts_train), ("valid", counts_valid), ("test", counts_test)]:
    for cid in range(NUM_CLASSES):
        if counts[cid] == 0:
            missing.append((split, CLASS_NAMES[cid]))

if missing:
    print("\nWARNING: some splits are missing classes:", missing)
    print("Training will continue but consider collecting more images for the rare classes.")
else:
    print("\nEvery split contains every stage.")

DATA_YAML = DATASET_DIR / "data.yaml"
data_cfg = {
    "path":  str(DATASET_DIR),
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    NUM_CLASSES,
    "names": CLASS_NAMES,
}
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print("\nWrote:", DATA_YAML)

## 5. Visualize class distribution per split

Quick sanity check: bar chart of object counts per stage in each split.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

split_counts = {
    "train": [counts_train[c] for c in range(NUM_CLASSES)],
    "valid": [counts_valid[c] for c in range(NUM_CLASSES)],
    "test":  [counts_test[c]  for c in range(NUM_CLASSES)],
}
x = np.arange(NUM_CLASSES)
width = 0.27
fig, ax = plt.subplots(figsize=(10, 4.5))
for i, (split, vals) in enumerate(split_counts.items()):
    ax.bar(x + (i - 1) * width, vals, width, label=split)
    for xi, v in zip(x + (i - 1) * width, vals):
        ax.text(xi, v + 0.5, str(v), ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=20, ha="right")
ax.set_ylabel("object count")
ax.set_title("Per-class object counts per split (after stratified re-split)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Train YOLO26m

Settings tuned for a small, imbalanced dataset on a Kaggle P100 (16 GB):
- **`yolo26m.pt`** — more capacity than `s` so rare classes get learned.
- **`imgsz=640`**.
- **`AdamW`** + **`cos_lr`** + `lr0=0.002` → stable on small data.
- Strong augmentation: **mosaic + mixup + copy-paste + HSV + rotation**.
- Higher **`cls=1.0`** weight pushes the model to discriminate stages.
- `epochs=200` with `patience=50` so it auto-stops if mAP plateaus.

> **If you hit CUDA OOM** (e.g. on T4 x2 with weird sharding): drop `BATCH` to **8** or `IMG_SIZE` to **512**, or switch `PRETRAINED` to `"yolo26s.pt"`.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

PRETRAINED   = "yolo26m.pt"
RUN_PROJECT  = "/kaggle/working/yolo26_runs"
RUN_NAME     = "yolo26m_640_stage_v2"
IMG_SIZE     = 640
EPOCHS       = 200
BATCH        = 16

print("Pretrained :", PRETRAINED)
print("Data       :", DATA_YAML)
print("Run        :", f"{RUN_PROJECT}/{RUN_NAME}")

model = YOLO(PRETRAINED)

results = model.train(
    task="detect",
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=50,
    device=DEVICE,
    project=RUN_PROJECT,
    name=RUN_NAME,
    exist_ok=True,

    optimizer="AdamW",
    lr0=0.002,
    lrf=0.01,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    momentum=0.937,

    box=7.5,
    cls=1.0,
    dfl=1.5,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.30,
    erasing=0.4,
    close_mosaic=10,

    multi_scale=True,
    dropout=0.10,
    seed=SEED,
    verbose=True,
    plots=True,
    save=True,
)

BEST_MODEL_PATH = str(Path(RUN_PROJECT) / RUN_NAME / "weights" / "best.pt")
print("\nBest model:", BEST_MODEL_PATH)

## 7. Evaluate on validation and test splits

Reports overall **and per-class** Precision / Recall / F1 / mAP@0.5 / mAP@0.5-0.95.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from ultralytics import YOLO

if not Path(BEST_MODEL_PATH).exists():
    raise FileNotFoundError(f"best.pt not found at {BEST_MODEL_PATH}")

best_model = YOLO(BEST_MODEL_PATH)

EVAL_DIR = Path(RUN_PROJECT) / RUN_NAME / "final_eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

def _safe_float(x):
    try:
        v = float(x)
        return v if not np.isnan(v) else None
    except Exception:
        return None

def _per_class_table(metrics, label):
    box = metrics.box
    p = np.atleast_1d(np.asarray(box.p, dtype=float))
    r = np.atleast_1d(np.asarray(box.r, dtype=float))
    map5095 = np.atleast_1d(np.asarray(box.maps, dtype=float)) if hasattr(box, "maps") else np.full(NUM_CLASSES, np.nan)
    try:
        ap50 = np.atleast_1d(np.asarray(box.ap50, dtype=float))
    except Exception:
        ap50 = np.full(NUM_CLASSES, np.nan)

    f1 = np.where((p + r) > 0, 2 * p * r / (p + r + 1e-12), 0.0)
    rows = []
    class_idx_seen = list(metrics.ap_class_index) if hasattr(metrics, "ap_class_index") else list(range(len(p)))
    for i, cid in enumerate(class_idx_seen):
        rows.append({
            "class": CLASS_NAMES[int(cid)],
            "P":     round(float(p[i]), 4),
            "R":     round(float(r[i]), 4),
            "F1":    round(float(f1[i]), 4),
            "mAP50": round(float(ap50[i]) if i < len(ap50) else float("nan"), 4),
            "mAP50-95": round(float(map5095[i]) if i < len(map5095) else float("nan"), 4),
        })
    df = pd.DataFrame(rows)
    mp = _safe_float(box.mp) or 0.0
    mr = _safe_float(box.mr) or 0.0
    overall = {
        "class":    "ALL",
        "P":        round(mp, 4),
        "R":        round(mr, 4),
        "F1":       round(2 * mp * mr / max(mp + mr, 1e-12), 4),
        "mAP50":    round(_safe_float(box.map50) or 0.0, 4),
        "mAP50-95": round(_safe_float(box.map) or 0.0, 4),
    }
    df = pd.concat([df, pd.DataFrame([overall])], ignore_index=True)
    print(f"\n=== {label} ===")
    print(df.to_string(index=False))
    return df

val_metrics = best_model.val(
    data=str(DATA_YAML), split="val", imgsz=IMG_SIZE, device=DEVICE,
    task="detect", plots=True, save_json=True, verbose=False,
    project=str(EVAL_DIR), name="val_eval", exist_ok=True,
)
test_metrics = best_model.val(
    data=str(DATA_YAML), split="test", imgsz=IMG_SIZE, device=DEVICE,
    task="detect", plots=True, save_json=True, verbose=False,
    project=str(EVAL_DIR), name="test_eval", exist_ok=True,
)

df_val  = _per_class_table(val_metrics,  "Validation metrics")
df_test = _per_class_table(test_metrics, "Test metrics")

df_val.to_csv(EVAL_DIR / "val_per_class.csv", index=False)
df_test.to_csv(EVAL_DIR / "test_per_class.csv", index=False)
print("\nSaved per-class tables to:", EVAL_DIR)

## 8. Show the confusion matrix and PR curves

Confirms each stage is being predicted correctly, not just one or two dominant ones.

In [ ]:
from pathlib import Path
from IPython.display import Image as IPyImage, Markdown, display

for label, sub in [("Validation", "val_eval"), ("Test", "test_eval")]:
    eval_sub = EVAL_DIR / sub
    display(Markdown(f"### {label} plots — `{eval_sub}`"))
    for img_name in [
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
        "PR_curve.png",
        "F1_curve.png",
        "P_curve.png",
        "R_curve.png",
    ]:
        img_path = eval_sub / img_name
        if img_path.exists():
            display(Markdown(f"**{img_name}**"))
            display(IPyImage(filename=str(img_path)))

## 9. Predict on the test set with TTA

Uses **Test-Time Augmentation** (`augment=True`) and a slightly lower confidence threshold for better recall on rare stages. Saves visualized predictions to `/kaggle/working/yolo26_predictions/`.

In [ ]:
from pathlib import Path

PRED_PROJECT = "/kaggle/working/yolo26_predictions"
PRED_NAME    = "test_predictions_tta"

test_images_dir = DATASET_DIR / "test" / "images"
if not test_images_dir.exists():
    raise FileNotFoundError(f"Test images dir missing: {test_images_dir}")

pred_results = best_model.predict(
    source=str(test_images_dir),
    imgsz=IMG_SIZE,
    conf=0.20,
    iou=0.50,
    augment=True,
    device=DEVICE,
    save=True,
    save_txt=True,
    save_conf=True,
    project=PRED_PROJECT,
    name=PRED_NAME,
    exist_ok=True,
    verbose=False,
)

PRED_DIR = Path(PRED_PROJECT) / PRED_NAME
print("Predictions saved at:", PRED_DIR)

## 10. `predict_stage()` helper — return the **dominant stage** for an image

`predict_stage(path)` runs the trained model on a single image and returns:

```python
{ "stage": "Flowering", "confidence": 0.87, "all_detections": [ ... ] }
```

The dominant stage is the one with the **highest summed confidence** across detections (more robust than just the top box).

In [ ]:
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt

def predict_stage(image_path, conf=0.20, iou=0.50, augment=True, show=True):
    """Run the trained YOLO26 model on one image and return the dominant orchid stage."""
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(image_path)

    res = best_model.predict(
        source=str(image_path),
        imgsz=IMG_SIZE,
        conf=conf,
        iou=iou,
        augment=augment,
        device=DEVICE,
        save=False,
        verbose=False,
    )[0]

    score_per_class = defaultdict(float)
    detections = []
    if res.boxes is not None and len(res.boxes) > 0:
        cls_ids = res.boxes.cls.cpu().numpy().astype(int)
        confs = res.boxes.conf.cpu().numpy().astype(float)
        xyxy = res.boxes.xyxy.cpu().numpy()
        for cid, c, box in zip(cls_ids, confs, xyxy):
            name = CLASS_NAMES[int(cid)]
            score_per_class[name] += float(c)
            detections.append({
                "class": name, "conf": round(float(c), 4),
                "box":  [round(float(v), 1) for v in box],
            })

    if score_per_class:
        stage = max(score_per_class, key=score_per_class.get)
        stage_conf = max(d["conf"] for d in detections if d["class"] == stage)
    else:
        stage, stage_conf = None, 0.0

    if show:
        plotted = res.plot()
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(plotted[..., ::-1])
        ax.set_title(f"{image_path.name}  ->  stage = {stage}  (conf={stage_conf:.2f})")
        ax.axis("off")
        plt.show()

    return {"stage": stage, "confidence": round(stage_conf, 4), "all_detections": detections}


sample_images = sorted((DATASET_DIR / "test" / "images").iterdir())[:6]
print("Predicting stage for first few test images:\n")
for img in sample_images:
    out = predict_stage(img, show=True)
    print(f"  {img.name}  ->  {out['stage']}  ({out['confidence']:.2f})")
    if out["all_detections"]:
        print("     all detections:", out["all_detections"])
    print()

## 11. Whole-test-set "stage classification" report

Compares the **predicted dominant stage** with the **ground-truth dominant stage** for every test image. Reports overall accuracy + per-stage accuracy.

In [ ]:
from collections import Counter
import pandas as pd
from pathlib import Path

test_imgs_dir = DATASET_DIR / "test" / "images"
test_lbls_dir = DATASET_DIR / "test" / "labels"

rows = []
for img in sorted(test_imgs_dir.iterdir()):
    if img.suffix.lower() not in IMG_EXTS:
        continue
    lbl = test_lbls_dir / (img.stem + ".txt")
    gt_classes = []
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            line = line.strip()
            if line:
                try:
                    gt_classes.append(int(line.split()[0]))
                except Exception:
                    pass
    gt_stage = CLASS_NAMES[Counter(gt_classes).most_common(1)[0][0]] if gt_classes else None

    pred = predict_stage(img, show=False)
    rows.append({
        "image":      img.name,
        "gt_stage":   gt_stage,
        "pred_stage": pred["stage"],
        "pred_conf":  pred["confidence"],
        "correct":    gt_stage is not None and gt_stage == pred["stage"],
    })

df_stage = pd.DataFrame(rows)
print(df_stage.to_string(index=False))

evaluable = df_stage.dropna(subset=["gt_stage"])
if len(evaluable) > 0:
    overall = evaluable["correct"].mean()
    print(f"\nOverall stage accuracy on test set: {overall*100:.2f}%   ({int(evaluable['correct'].sum())}/{len(evaluable)})")
    per_stage = evaluable.groupby("gt_stage")["correct"].agg(["mean", "count"]).rename(columns={"mean": "accuracy"})
    per_stage["accuracy"] = (per_stage["accuracy"] * 100).round(2)
    print("\nPer-stage accuracy:")
    print(per_stage.to_string())

df_stage.to_csv(EVAL_DIR / "test_stage_classification.csv", index=False)
print("\nSaved:", EVAL_DIR / "test_stage_classification.csv")

## 12. Bundle outputs into one zip in `/kaggle/working/`

After **Save & Run All (Commit)** finishes, this zip will be in the **Output** tab of your notebook (right sidebar) — click it and download. The zip contains `best.pt`, all metrics, predictions, and the resolved `data.yaml`.

In [ ]:
from pathlib import Path
import shutil
import zipfile

run_dir   = Path(RUN_PROJECT) / RUN_NAME
pred_dir  = Path(PRED_PROJECT) / PRED_NAME
eval_dir  = EVAL_DIR

EXPORT_ZIP = Path("/kaggle/working/yolo26_stage_v2_output.zip")
if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

best_on_disk = run_dir / "weights" / "best.pt"
if best_on_disk.is_file():
    print("best.pt :", best_on_disk, "|", round(best_on_disk.stat().st_size / (1024 * 1024), 2), "MB")
else:
    print("WARNING: best.pt not found at", best_on_disk)

with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for prefix, folder in [
        ("runs", run_dir),
        ("predictions", pred_dir),
        ("metrics_eval", eval_dir),
    ]:
        if not Path(folder).exists():
            print("skip (missing):", folder)
            continue
        for fp in Path(folder).rglob("*"):
            if fp.is_file():
                zf.write(fp, arcname=(Path(prefix) / fp.relative_to(folder)).as_posix())
    if Path(DATA_YAML).exists():
        zf.write(DATA_YAML, arcname="data.yaml")

# Also copy best.pt as a standalone file in /kaggle/working/ so it shows up directly in the Output tab.
if best_on_disk.is_file():
    shutil.copy2(best_on_disk, Path("/kaggle/working/best.pt"))
    print("Copied best.pt -> /kaggle/working/best.pt")

print("\nzip          :", EXPORT_ZIP)
print("zip size MB  :", round(EXPORT_ZIP.stat().st_size / (1024 * 1024), 2))
print("\nDownload from the right-sidebar 'Output' tab of this notebook after Save & Run All completes.")